# Phase 5: Baseline ML Model Training & Comparative Evaluation
## AI-Driven Intelligent Greywater Management and Smart Reuse Routing System

### Section 1: Project Objective
The objective of Phase 5 is to train and evaluate baseline supervised machine learning models to classify greywater into four operational routing destinations:
- **Class 0:** Sewer Bypass (Critical safety cutoff for toxic/high-pathogen greywater)
- **Class 1:** Bio-filtration (Secondary biological treatment)
- **Class 2:** Restricted Irrigation (Direct non-food irrigation / landscaping)
- **Class 3:** Indoor Reuse (Toilet flushing & non-potable domestic circulation)

Models evaluated:
1. **XGBoost (Primary Model):** Gradient boosted decision tree multiclass classifier.
2. **Random Forest (Baseline Model):** Bagged ensemble decision tree classifier.

*Note: In accordance with Phase 5 guidelines, no automated hyperparameter tuning (GridSearchCV/Optuna) is performed at this stage. We evaluate baseline architectures, safety error distributions, and cross-validation stability.*


### Section 2: Load Processed Data
Load the stratified train, validation, and test splits prepared in Phase 4.


In [ ]:
import os
import sys
import types
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Windows Application Control patch for sklearn C-extensions
m1 = types.ModuleType('sklearn.metrics.cluster._expected_mutual_info_fast')
m1.expected_mutual_information = lambda *a, **kw: 0.0
sys.modules['sklearn.metrics.cluster._expected_mutual_info_fast'] = m1

m2 = types.ModuleType('sklearn.linear_model._sgd_fast')
class DummyLoss: pass
m2.EpsilonInsensitive = DummyLoss
m2.Hinge = DummyLoss
m2.ModifiedHuber = DummyLoss
m2.SquaredEpsilonInsensitive = DummyLoss
m2.SquaredHinge = DummyLoss
m2._plain_sgd32 = lambda *a, **kw: None
m2._plain_sgd64 = lambda *a, **kw: None
sys.modules['sklearn.linear_model._sgd_fast'] = m2

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from xgboost import XGBClassifier

train_df = pd.read_csv('../dataset/processed/train.csv')
val_df = pd.read_csv('../dataset/processed/validation.csv')
test_df = pd.read_csv('../dataset/processed/test.csv')

meta_val = pd.read_csv('../dataset/processed/metadata_val.csv')
meta_test = pd.read_csv('../dataset/processed/metadata_test.csv')

print(f"Train split shape: {train_df.shape}")
print(f"Validation split shape: {val_df.shape}")
print(f"Test split shape: {test_df.shape}")


### Section 3: Verify Target Classes & Feature Separation
Verify feature consistency across all splits and separate features ($X$) and target ($y$).


In [ ]:
target_col = 'Routing_Class'
feature_cols = [c for c in train_df.columns if c != target_col]

assert list(train_df.columns) == list(val_df.columns) == list(test_df.columns), "Feature mismatch!"

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

class_names = {
    0: 'Sewer Bypass',
    1: 'Bio-filtration',
    2: 'Restricted Irrigation',
    3: 'Indoor Reuse'
}

print(f"Total model features: {len(feature_cols)}")
print(f"Features list: {feature_cols}")


### Section 4: Training, Validation, and Test Distribution
Examine the class distributions across the three datasets.


In [ ]:
dist_summary = pd.DataFrame({
    'Train_Count': y_train.value_counts().sort_index(),
    'Train_Pct': (y_train.value_counts(normalize=True).sort_index() * 100).round(2),
    'Val_Count': y_val.value_counts().sort_index(),
    'Val_Pct': (y_val.value_counts(normalize=True).sort_index() * 100).round(2),
    'Test_Count': y_test.value_counts().sort_index(),
    'Test_Pct': (y_test.value_counts(normalize=True).sort_index() * 100).round(2)
})
dist_summary.index = [f"Class {i}: {class_names[i]}" for i in dist_summary.index]
print(dist_summary)


### Section 5: Random Forest Baseline
Train a 300-tree baseline Random Forest classifier.
We specify `class_weight='balanced'` because Class 3 represents only 0.40% of the dataset.


In [ ]:
rf_baseline = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
rf_baseline.fit(X_train, y_train)
print("Random Forest baseline trained successfully.")


### Section 6: XGBoost Baseline
Train a 300-estimator XGBoost multiclass model using `multi:softprob`.


In [ ]:
xgb_baseline = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss"
)
xgb_baseline.fit(X_train, y_train)
print("XGBoost baseline trained successfully.")


### Section 7: Validation Metrics
Evaluate predictions on the validation set using Macro and Weighted averages as well as per-class breakdown.


In [ ]:
rf_val_preds = rf_baseline.predict(X_val)
rf_val_probs = rf_baseline.predict_proba(X_val)

xgb_val_preds = xgb_baseline.predict(X_val)
xgb_val_probs = xgb_baseline.predict_proba(X_val)

def get_metrics_dict(y_true, y_pred, model_name):
    return {
        "model": model_name,
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "macro_precision": round(precision_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "macro_recall": round(recall_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "macro_f1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "weighted_precision": round(precision_score(y_true, y_pred, average="weighted", zero_division=0), 4),
        "weighted_recall": round(recall_score(y_true, y_pred, average="weighted", zero_division=0), 4),
        "weighted_f1": round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4)
    }

val_comparison = pd.DataFrame([
    get_metrics_dict(y_val, rf_val_preds, "Random Forest"),
    get_metrics_dict(y_val, xgb_val_preds, "XGBoost")
])
print("Validation Model Metrics Comparison:")
print(val_comparison.to_string(index=False))

print("\nRandom Forest Detailed Classification Report:")
print(classification_report(y_val, rf_val_preds, target_names=[class_names[i] for i in range(4)], zero_division=0))

print("\nXGBoost Detailed Classification Report:")
print(classification_report(y_val, xgb_val_preds, target_names=[class_names[i] for i in range(4)], zero_division=0))


### Section 8: Confusion Matrices
Visualize classification performance and routing errors via confusion matrices.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = [0, 1, 2, 3]
tick_labels = ["Sewer Bypass", "Bio-filtration", "Restricted Irrigation", "Indoor Reuse"]

cm_rf = confusion_matrix(y_val, rf_val_preds, labels=labels)
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues", ax=axes[0], xticklabels=tick_labels, yticklabels=tick_labels)
axes[0].set_title("Random Forest — Validation Confusion Matrix")
axes[0].set_xlabel("Predicted Class")
axes[0].set_ylabel("True Class")
axes[0].tick_params(axis='x', rotation=45)

cm_xgb = confusion_matrix(y_val, xgb_val_preds, labels=labels)
sns.heatmap(cm_xgb, annot=True, fmt="d", cmap="Greens", ax=axes[1], xticklabels=tick_labels, yticklabels=tick_labels)
axes[1].set_title("XGBoost — Validation Confusion Matrix")
axes[1].set_xlabel("Predicted Class")
axes[1].set_ylabel("True Class")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


### Section 9: Safety-Focused Error Analysis
Operational safety is paramount. Missing a Sewer Bypass case (False Negative) routes contaminated or pathogen-rich water into human or bio-filter systems, risking biological fouling, soil contamination, or infection.
Conversely, false alarms (routing clean water to sewer) only incur slight water volume loss.


In [ ]:
def calc_safety(y_true, y_pred, model_name):
    true_bp = (y_true == 0)
    pred_bp = (y_pred == 0)
    tp = np.sum(true_bp & pred_bp)
    fn = np.sum(true_bp & (~pred_bp))
    fp = np.sum((~true_bp) & pred_bp)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    return {
        "model": model_name,
        "true_bypass_count": int(np.sum(true_bp)),
        "true_positives": int(tp),
        "dangerous_false_negatives": int(fn),
        "false_alarm_positives": int(fp),
        "sewer_bypass_recall": round(float(recall), 4),
        "sewer_bypass_precision": round(float(prec), 4)
    }

safety_val = pd.DataFrame([
    calc_safety(y_val, rf_val_preds, "Random Forest"),
    calc_safety(y_val, xgb_val_preds, "XGBoost")
])
print("Safety-Focused Error Analysis on Validation Set:")
print(safety_val.to_string(index=False))


### Section 10: 5-Fold Stratified Cross-Validation
Perform cross-validation strictly on the training set to measure model stability across unseen training folds.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_accs, rf_mf1s, rf_wf1s = [], [], []
xgb_accs, xgb_mf1s, xgb_wf1s = [], [], []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr_f, y_tr_f = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
    X_va_f, y_va_f = X_train.iloc[va_idx], y_train.iloc[va_idx]
    
    # RF
    rf_f = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced")
    rf_f.fit(X_tr_f, y_tr_f)
    p_rf = rf_f.predict(X_va_f)
    rf_accs.append(accuracy_score(y_va_f, p_rf))
    rf_mf1s.append(f1_score(y_va_f, p_rf, average="macro", zero_division=0))
    rf_wf1s.append(f1_score(y_va_f, p_rf, average="weighted", zero_division=0))
    
    # XGB
    xgb_f = XGBClassifier(
        objective="multi:softprob", num_class=4, n_estimators=300, max_depth=4,
        learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric="mlogloss"
    )
    xgb_f.fit(X_tr_f, y_tr_f)
    p_xgb = xgb_f.predict(X_va_f)
    xgb_accs.append(accuracy_score(y_va_f, p_xgb))
    xgb_mf1s.append(f1_score(y_va_f, p_xgb, average="macro", zero_division=0))
    xgb_wf1s.append(f1_score(y_va_f, p_xgb, average="weighted", zero_division=0))

cv_df = pd.DataFrame([
    {
        "model": "Random Forest",
        "mean_accuracy": round(np.mean(rf_accs), 4),
        "std_accuracy": round(np.std(rf_accs), 4),
        "mean_macro_f1": round(np.mean(rf_mf1s), 4),
        "std_macro_f1": round(np.std(rf_mf1s), 4),
        "mean_weighted_f1": round(np.mean(rf_wf1s), 4),
        "std_weighted_f1": round(np.std(rf_wf1s), 4)
    },
    {
        "model": "XGBoost",
        "mean_accuracy": round(np.mean(xgb_accs), 4),
        "std_accuracy": round(np.std(xgb_accs), 4),
        "mean_macro_f1": round(np.mean(xgb_mf1s), 4),
        "std_macro_f1": round(np.std(xgb_mf1s), 4),
        "mean_weighted_f1": round(np.mean(xgb_wf1s), 4),
        "std_weighted_f1": round(np.std(xgb_wf1s), 4)
    }
])
print("5-Fold Cross-Validation Results (Training Set Only):")
print(cv_df.to_string(index=False))


### Section 11: Single Touchpoint Test Set Evaluation
The test set remained completely untouched during model training and validation. We now perform a single evaluation on the test set.


In [ ]:
rf_test_preds = rf_baseline.predict(X_test)
rf_test_probs = rf_baseline.predict_proba(X_test)

xgb_test_preds = xgb_baseline.predict(X_test)
xgb_test_probs = xgb_baseline.predict_proba(X_test)

test_comparison = pd.DataFrame([
    get_metrics_dict(y_test, rf_test_preds, "Random Forest (Test)"),
    get_metrics_dict(y_test, xgb_test_preds, "XGBoost (Test)")
])
print("Test Set Overall Performance:")
print(test_comparison.to_string(index=False))

print("\nTest Set Safety Metrics:")
test_safety = pd.DataFrame([
    calc_safety(y_test, rf_test_preds, "Random Forest (Test)"),
    calc_safety(y_test, xgb_test_preds, "XGBoost (Test)")
])
print(test_safety.to_string(index=False))


### Section 12: Probability Analysis
Inspect the predicted class probability distributions for validation and test samples.


In [ ]:
pred_sample = pd.DataFrame({
    "Sample_ID": meta_val["Sample_ID"].head(10),
    "Actual": [class_names[c] for c in y_val.head(10)],
    "Predicted": [class_names[c] for c in xgb_val_preds[:10]],
    "P(Sewer)": np.round(xgb_val_probs[:10, 0], 3),
    "P(BioFilt)": np.round(xgb_val_probs[:10, 1], 3),
    "P(Irrig)": np.round(xgb_val_probs[:10, 2], 3),
    "P(Indoor)": np.round(xgb_val_probs[:10, 3], 3),
})
print("Sample Validation Probability Outputs (XGBoost):")
print(pred_sample.to_string(index=False))


### Section 13: Model Confidence Analysis
Evaluate prediction confidence (maximum softmax probability) and identify borderline samples with confidence $< 0.70$.


In [ ]:
def conf_stats(probs, preds, y_true, model_name, split):
    max_p = np.max(probs, axis=1)
    low_mask = max_p < 0.70
    corr = (preds == y_true)
    return {
        "model": model_name,
        "split": split,
        "mean_confidence": round(float(np.mean(max_p)), 4),
        "median_confidence": round(float(np.median(max_p)), 4),
        "min_confidence": round(float(np.min(max_p)), 4),
        "low_conf_count": int(np.sum(low_mask)),
        "low_conf_pct": round(float(np.sum(low_mask) / len(max_p) * 100), 2),
        "acc_on_low_conf": round(float(np.mean(corr[low_mask])) if np.sum(low_mask) > 0 else 1.0, 4)
    }

conf_table = pd.DataFrame([
    conf_stats(xgb_val_probs, xgb_val_preds, y_val, "XGBoost", "Val"),
    conf_stats(rf_val_probs, rf_val_preds, y_val, "Random Forest", "Val"),
    conf_stats(xgb_test_probs, xgb_test_preds, y_test, "XGBoost", "Test"),
    conf_stats(rf_test_probs, rf_test_preds, y_test, "Random Forest", "Test"),
])
print("Model Confidence Summary:")
print(conf_table.to_string(index=False))


### Section 14: Feature Importance
Analyze which water quality parameters and source indicators drive routing decisions in each model.
*Note: Feature importance represents statistical association within the decision trees, not causal mechanisms.*


In [ ]:
rf_imp = pd.Series(rf_baseline.feature_importances_, index=feature_cols).sort_values(ascending=False)
xgb_imp = pd.Series(xgb_baseline.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

rf_imp.head(10).plot(kind='barh', ax=axes[0], color='#3b82f6')
axes[0].set_title("Random Forest Top 10 Features (MDI Impurity)")
axes[0].invert_yaxis()

xgb_imp.head(10).plot(kind='barh', ax=axes[1], color='#10b981')
axes[1].set_title("XGBoost Top 10 Features (Gain)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


### Section 15: Model Comparison & Synthesis
Synthesize validation vs. test metrics and assess operational tradeoffs.


In [ ]:
comparison_all = pd.concat([
    val_comparison.assign(split="Validation"),
    test_comparison.assign(split="Test")
])
print(comparison_all[['split', 'model', 'accuracy', 'macro_f1', 'weighted_f1']].to_string(index=False))


### Section 16: Initial Conclusions & Scientific Limitations
1. **Model Performance:** Both models learn the operational routing logic effectively, achieving $>95\%$ overall accuracy and $>0.94$ weighted F1 score.
2. **Safety Divergence:** While XGBoost achieved higher overall test accuracy (97.33% vs 96.44%), Random Forest demonstrated a higher test recall on Sewer Bypass (98.51% with 1 false negative vs. XGBoost's 95.52% with 3 false negatives).
3. **Severe Class Imbalance:** Class 3 (Indoor Reuse) represents only 0.40% (6 total samples across the dataset). Neither baseline model predicted Class 3 on validation or test sets, suppressing macro F1 scores to ~0.72 - 0.73.
4. **Important Scientific Limitation:**
   > *"The supervised models are currently learning rule-derived operational routing labels. Their performance indicates agreement with the labeling framework rather than independent validation against experimentally measured expert ground truth."*
5. **Next Steps:** Hyperparameter tuning, class-weight calibration, and safety-cost penalty matrix optimization will be explored in subsequent optimization phases.
